# 2) Statistical models — Exercises

**Goals:** find the heating and cooling thresholds from data, quantify the weather leak, test how holidays should enter a regression, build the error correction for an intraday forecast, and put a number on the value of a better temperature forecast.

The setup cell loads the lecture's data with the honest temperature forecast, the `daytype`, `hour` and `month` columns, the lecture's model formula `LECTURE` and a helper `mae`. Demand in MW, temperatures in °C.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# forecast_data.py lives in the module folder: search upwards so this works from exercise/ and exercise/solutions/
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "forecast_data.py").exists())))
from forecast_data import load_data, temperature_forecast

warnings.filterwarnings("ignore")
df = load_data()
df["temperature_forecast"] = temperature_forecast(df)
dow = df.index.dayofweek
df["daytype"] = np.where(df.holiday | (dow == 6), "sunday_or_holiday", np.where(dow == 5, "saturday", "workday"))
df["hour"], df["month"] = df.index.hour, df.index.month
train, test = df.loc[:"2024"], df.loc["2025"]
SPLINE = "bs(temperature, df=6, lower_bound=-12, upper_bound=40)"
LECTURE = f"demand ~ C(hour):C(daytype) + C(month) + {SPLINE}"     # the lecture's temperature model


def mae(forecast, actual):
    return float(np.abs(np.asarray(forecast) - np.asarray(actual)).mean())

### Warm-ups

**1. Finding the thresholds**

The lecture's spline learned the temperature response without being told where heating and cooling start. Do it explicitly: for heating thresholds in [12, 14, 16, 18] °C and cooling thresholds in [20, 22, 24, 26] °C, fit `demand ~ C(hour):C(daytype) + C(month) + hdd + cdd` (hdd = max(threshold_h − T, 0), cdd = max(T − threshold_c, 0)) on **2023**, and score it on **2024** with the actual temperature. Write `best_thresholds()` returning the best pair and the table of validation MAEs.

In [ ]:
def best_thresholds(heating=(12, 14, 16, 18), cooling=(20, 22, 24, 26)) -> tuple[tuple[int, int], pd.DataFrame]:
    """Heating and cooling thresholds (deg C) with the lowest 2024 MAE, and the full MAE table."""
    ...  # YOUR CODE HERE

In [ ]:
best, table = best_thresholds()
assert best == (16, 22), "the data were generated with heating below 16 and cooling above 22 deg C"
print("best thresholds:", best)
table.round(1)

**2. How much does the weather leak flatter?**

Write `leak_optimism(formula)`: fit the formula on 2023–2024, score 2025 once with the temperature forecast and once with the actual temperature, and return both MAEs and how much better the leaky score looks (in %). Apply it to the lecture's model and to the calendar-only model.

In [ ]:
def leak_optimism(formula: str) -> dict:
    """Honest and leaky 2025 MAE of a model, and the leak's apparent improvement in %."""
    ...  # YOUR CODE HERE

In [ ]:
weather = leak_optimism(LECTURE)
calendar = leak_optimism("demand ~ C(hour):C(daytype) + C(month)")
assert np.isclose(weather["honest"], 73.6, atol=0.1) and np.isclose(weather["leaky"], 58.6, atol=0.1)
assert np.isclose(calendar["looks_better_by_%"], 0.0), "a model without temperature cannot leak through it"
pd.DataFrame({"temperature model": weather, "calendar only": calendar}).round(1)

### Core

**3. How should holidays enter a regression?**

The lecture treats holidays as Sundays. Compare three versions of the lecture's model, fitted on 2023–2024 and scored on 2025 (honest temperature) — overall and on holidays only: (a) holidays ignored (they count as normal weekdays), (b) holidays as Sundays (the lecture), (c) holidays as a fourth day type of their own. Write `holiday_variants()`.

In [ ]:
def holiday_variants() -> pd.DataFrame:
    """2025 MAE overall and on holidays for three ways of encoding holidays."""
    ...  # YOUR CODE HERE

In [ ]:
variants = holiday_variants()
assert variants.loc["ignored", "MAE holidays"] > 2 * variants.loc["as Sundays (lecture)", "MAE holidays"]
assert np.isclose(variants.loc["as Sundays (lecture)", "MAE all"], 73.6, atol=0.1)
variants.round(1)

**4. An intraday forecast**

Write `corrected_forecast(base, known_error, phi, k)` — the lecture's correction, base + φᵏ · (error known k hours earlier). Then build an **intraday** forecast: issued at 06:00, for the hours 07:00–12:00 of the same day (k = 1 … 6). Compare its MAE on 2025 with the day-ahead forecast for the same hours.

In [ ]:
def corrected_forecast(base: pd.Series, known_error: pd.Series, phi: float, k) -> pd.Series:
    """Model forecast plus the error known k hours earlier, shrunk by phi**k (k may vary by hour)."""
    ...  # YOUR CODE HERE

In [ ]:
model = smf.ols(LECTURE, train).fit()
phi = (df.demand - model.predict(df)).loc[:"2024"].autocorr(1)
known_error = df.demand - model.predict(df)
morning = test.index[(test.index.hour >= 7) & (test.index.hour <= 12)]
base = model.predict(df.assign(temperature=df.temperature_forecast)).loc[morning]
intraday = corrected_forecast(base, known_error, phi, morning.hour - 6)       # issued at 06:00: k = 1 ... 6
toy = corrected_forecast(pd.Series([100.0], index=[pd.Timestamp("2024-01-01 05:00")]),
                         pd.Series([10.0], index=[pd.Timestamp("2024-01-01 03:00")]), 0.5, 2)
assert np.isclose(toy.iloc[0], 100 + 0.25 * 10)
result = pd.Series({"day-ahead": mae(base, df.demand.loc[morning]), "intraday from 06:00": mae(intraday, df.demand.loc[morning])})
assert result["intraday from 06:00"] < 0.9 * result["day-ahead"]
result.round(1)

**5. Interpretation**

Which holiday encoding wins, and why does "ignored" fail so badly on holidays? How large is the intraday gain for the morning hours, and why could a TSO not simply use the intraday forecast for the day-ahead market?

*Your answer here.*

### Challenge

**6. What is a better weather forecast worth?**

Scale the temperature-forecast error: use temperature + s · (forecast − temperature) for s in [0, 0.25, 0.5, 1, 1.5, 2] (s = 0 is a perfect forecast, s = 1 the lecture's). Keep the model fitted on actual temperatures, score 2025 for each s, and report the demand MAE against the temperature MAE. Large errors can push a forecast outside the range the spline was fitted on (−12 to 40 °C), where it cannot extrapolate: clip the forecast temperatures to that range first — the model then treats an extreme forecast like the most extreme value it has learned. Roughly how many MW of demand error does each °C of temperature error cost?

In [ ]:
def weather_value(scales=(0, 0.25, 0.5, 1, 1.5, 2)) -> pd.DataFrame:
    """2025 demand MAE (MW) for temperature forecasts with scaled errors."""
    ...  # YOUR CODE HERE

In [ ]:
value = weather_value()
assert value.demand_MAE.is_monotonic_increasing
assert np.isclose(value.loc[1, "demand_MAE"], 73.6, atol=0.1) and np.isclose(value.loc[0, "demand_MAE"], 58.6, atol=0.1)
slope = np.polyfit(value.temperature_MAE.iloc[2:], value.demand_MAE.iloc[2:], 1)[0]
print(f"beyond small errors, each extra 1 deg C of temperature MAE costs about {slope:.0f} MW of demand MAE")
value.round(2)

**7. Interpretation**

Describe the curve: is the cost of temperature error linear, and why not at small errors? What does the number suggest a TSO should be willing to pay for better weather forecasts — and what else would you need to know to answer that properly?

*Your answer here.*